# 1.7 分數怎麼變成機率？

# 第 1 章：從猜下一個字開始

前置：讀過入門指南的 Python 與 tensor 部分即可。這一章只用幾個字；不要先開完整 Transformer。每段 code 都可單獨執行。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：把分數換成比例：像分一塊餅**

一起加同常數不改比例，分數差才重要。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/softmax.svg")))

**先想一想：**所有分數加 100，會讓模型更有信心嗎？

softmax 把較高分數變成較高機率。所有分數一起加同一常數不改相對差異，因此不改分佈；先減最大值可避免 exponential 太大。

**把直覺寫成數學：**$p_i=e^{z_i-m}/\sum_j e^{z_j-m}$，$m=\max z$。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
z = torch.tensor([1.0, 2.0, -1.0])
p = (z - z.max()).exp()
p = p / p.sum()
print(p)
assert torch.allclose(p, z.softmax(0))
assert torch.allclose(p, (z + 100).softmax(0))

**如何讀結果：**只看分數差；總和約等於 1。浮點數計算用 allclose，不要求逐 bit 相同。

**只改一件事：**只提高第一個分數，預測另外兩個機率如何變。
